In [42]:
from operator import add
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, END, START
from pydantic import BaseModel, Field
from typing import Annotated,TypedDict

In [43]:
OLLAMA_MODEL = "qwen2.5-coder:14b"
OLLAMA_BASE_URL = "http://localhost:11434"
model = ChatOllama(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0,
)

In [44]:
essay_input={'essay':"""
Global warming refers to the long-term increase in Earth’s average surface temperature due to human activities, particularly the emission of greenhouse gases. Over the past century, industrialization, deforestation, and the burning of fossil fuels such as coal, oil, and natural gas have significantly increased the concentration of carbon dioxide and other heat-trapping gases in the atmosphere. This has intensified the natural greenhouse effect, causing the planet to warm at an unprecedented rate.

One of the most visible consequences of global warming is the melting of polar ice caps and glaciers. As temperatures rise, ice sheets in regions like Antarctica and Greenland are shrinking, contributing to rising sea levels. This poses a serious threat to coastal communities and island nations, where flooding and erosion are becoming more frequent. Additionally, warmer oceans lead to coral bleaching, which damages marine ecosystems and threatens biodiversity.

Global warming also affects weather patterns. Many regions are experiencing more intense and frequent extreme weather events, such as heatwaves, hurricanes, droughts, and heavy rainfall. These changes disrupt agriculture, reduce crop yields, and can lead to food shortages. In some parts of the world, prolonged droughts have caused water scarcity, while in others, excessive rainfall has led to flooding and infrastructure damage.

Another major concern is the impact on ecosystems and wildlife. Many species are unable to adapt quickly enough to the changing climate, leading to shifts in migration patterns, habitat loss, and even extinction. Forests, which act as important carbon sinks, are also under threat due to increased wildfires and deforestation. This creates a vicious cycle, as fewer trees mean less carbon dioxide is absorbed from the atmosphere, further accelerating global warming.

Human health is also at risk. Rising temperatures can lead to heat-related illnesses and increase the spread of diseases carried by insects, such as malaria and dengue fever. Air quality may worsen due to higher levels of pollution and allergens, affecting people with respiratory conditions. Vulnerable populations, especially in developing countries, are likely to suffer the most due to limited resources and infrastructure.

Despite these challenges, there are ways to mitigate global warming. Transitioning to renewable energy sources such as solar, wind, and hydroelectric power can significantly reduce greenhouse gas emissions. Energy efficiency, sustainable agriculture, and reforestation efforts also play a crucial role. On an individual level, people can contribute by conserving energy, reducing waste, and adopting environmentally friendly habits.

In conclusion, global warming is a complex and urgent issue that affects every aspect of life on Earth. While its impacts are already being felt, collective action from governments, organizations, and individuals can help slow its progression. Addressing global warming requires a combination of scientific innovation, policy changes, and a commitment to sustainable living to ensure a healthier planet for future generations.

"""}

# Thing to keep in mind

Yahan ph hu logo nh aik sh zyada scores store krny thy for that we need to add a reducer kiu k age hum normal as it is rehnay detay

```scores:Annotated[list[int]]```

tu is sh jb pehli chain chlti tu aa jata lets say [8] dusri sh b kxh number aa jatay but wo pehly wala ko replace kr daita we didnt want that

we wanted k teeno save houn tu is liye we had to do

```scores:Annotated[list[int],add]```

is sh kia hua k ab pehly walay ka score ata ha [8] dusry walay ka [6] teesra ka [9] tu in lists ko add/combine kr k aik list bn jaati ha
[8,6,9] ki

is liye when returning output we did

```'scores':[int(response.score)]```

kiu k srf ```'scores':response.score``` krny sh hamain mil 8 6 9 ,,, jisy hum concatinate kr k aik list nahi bna skty thy


In [46]:
class EssayState(TypedDict):
    essay:str
    cot_feedback:str
    doa_feedback:str
    language_feedback:str

    scores:Annotated[list[int],add]

    finalscore:int

In [45]:
class CotState(BaseModel):
    feedback:str=Field(description="A small comment after the analysis of the essay based on how well it is written in terms of chain of thought")
    score:int=Field(description="The score based on your analysis",ge=0,le=10)


class DoaState(BaseModel):
    feedback:str=Field(description="A small comment after the analysis of the essay based on how well it is written in terms of Depth of Analysis")
    score:int=Field(description="The score based on your analysis",ge=0,le=10)


class LanguageState(BaseModel):
    feedback:str=Field(description="A small comment after the analysis of the essay based on how well it is written in terms of Language used")
    score:int=Field(description="The score based on your analysis",ge=0,le=10)

In [47]:
def RateCot(state:EssayState):
    template=PromptTemplate(
        input_variables=['essay'],
        template="You need to rate the essay on the basis of chain of thought and output a feedback and rank it from 1 to 10. Essay= {essay} "
    )

    structured_model=model.with_structured_output(CotState)
    prompt=template.format(essay=state['essay'])

    response=structured_model.invoke(prompt)

    return {'cot_feedback':response.feedback,'scores':[int(response.score)]}


In [48]:
def RateDoa(state:EssayState):
    template=PromptTemplate(
        input_variables=['essay'],
        template="You need to rate the essay on the basis of Depth of Analysis and output a feedback and rank it from 1 to 10. Essay= {essay} "
    )

    structured_model=model.with_structured_output(DoaState)
    prompt=template.format(essay=state['essay'])

    response=structured_model.invoke(prompt)

    return {'doa_feedback':response.feedback,'scores':[int(response.score)]}


In [49]:
def RateLanguage(state:EssayState):
    template=PromptTemplate(
        input_variables=['essay'],
        template="You need to rate the essay on the basis of Language Used and output a feedback and rank it from 1 to 10. Essay= {essay} "
    )

    structured_model=model.with_structured_output(LanguageState)
    prompt=template.format(essay=state['essay'])

    response=structured_model.invoke(prompt)

    return {'language_feedback':response.feedback,'scores':[int(response.score)]}


In [50]:
graph=StateGraph(EssayState)

graph.add_node('RateCot',RateCot)
graph.add_node('RateDoa',RateDoa)
graph.add_node('RateLanguage',RateLanguage)

graph.add_edge(START,'RateCot')
graph.add_edge(START,'RateDoa')
graph.add_edge(START,'RateLanguage')
graph.add_edge('RateCot',END)
graph.add_edge('RateDoa',END)
graph.add_edge('RateLanguage',END)

workflow=graph.compile()

In [56]:
final=workflow.invoke(essay_input)


In [60]:
print(final["scores"])

print(final["language_feedback"])

[8, 8, 7]
The essay is well-written with clear and concise language. It effectively communicates the causes, effects, and potential solutions related to global warming. The structure is logical, and the information is presented in a coherent manner. However, there are some areas where the writing could be improved for better engagement and impact. For instance, using more vivid examples or statistics could enhance the persuasiveness of the argument. Additionally, incorporating more diverse perspectives or counterarguments would strengthen the overall analysis. Overall, the essay provides a solid foundation for understanding global warming but has room for enhancement in terms of depth and presentation.
